# Expense claims review

Answers three questions: approved spend per department, rejection rate per category, and claims above their category's policy limit.

**How to use:** change `FILE_PATH` in the first code cell, then run all cells (Run > Run All Cells).

Every row is kept. Rows with problems are marked, not removed. A marked row is left out of an answer only when the problem makes a value that answer needs unusable (for example an Amount that is not a number). Each answer says how many rows it used.

In [1]:
# The only line you need to change: the path to your CSV file.
FILE_PATH = "data/practice_expense_claims_synthetic.csv"

In [2]:
import pandas as pd
import plotly.express as px

# Policy limit per category (a claim is "above limit" if Amount is greater than this).
POLICY_LIMITS = {"Travel": 1500, "Meals": 75, "Software": 500, "Equipment": 1200, "Office Supplies": 200}

# Status values we expect to see.
EXPECTED_STATUSES = ["Approved", "Rejected", "Pending"]

# Columns the file must contain.
EXPECTED_COLUMNS = ["Claim ID", "Submitted", "Employee", "Employee ID", "Department",
                    "Category", "Amount", "Status", "Decided"]

## 1. Load the file

In [3]:
# Read everything as plain text first, so we can check types ourselves.
raw = pd.read_csv(FILE_PATH, dtype=str, keep_default_na=False)

# Stop early with a clear message if a column is missing.
absent = [c for c in EXPECTED_COLUMNS if c not in raw.columns]
if absent:
    raise ValueError(f"These expected columns are not in the file: {absent}")

# Remove stray spaces and treat blank cells as missing.
raw = raw[EXPECTED_COLUMNS].apply(lambda col: col.str.strip())
raw = raw.mask(raw == "")

print(f"Rows loaded: {len(raw)}")

Rows loaded: 200


## 2. Validate (rows are marked, never removed)

In [4]:
claims = raw.copy()

# Convert to proper types. Anything that cannot be converted becomes empty (NaN / NaT).
claims["amount_num"] = pd.to_numeric(claims["Amount"], errors="coerce")
# Dates must look like 2026-03-27 (year-month-day).
claims["submitted_dt"] = pd.to_datetime(claims["Submitted"], errors="coerce", format="%Y-%m-%d")
claims["decided_dt"] = pd.to_datetime(claims["Decided"], errors="coerce", format="%Y-%m-%d")

# --- Problem checks: each is True for a row that has that problem ---
missing_by_col = {c: claims[c].isna() for c in EXPECTED_COLUMNS}
any_missing = pd.concat(missing_by_col, axis=1).any(axis=1)

# Wrong type = a value is present but cannot be read as a number / date.
bad_amount = claims["Amount"].notna() & claims["amount_num"].isna()
bad_submitted = claims["Submitted"].notna() & claims["submitted_dt"].isna()
bad_decided = claims["Decided"].notna() & claims["decided_dt"].isna()
any_wrong_type = bad_amount | bad_submitted | bad_decided

# Unexpected values in the two "list" columns.
bad_category = claims["Category"].notna() & ~claims["Category"].isin(POLICY_LIMITS.keys())
bad_status = claims["Status"].notna() & ~claims["Status"].isin(EXPECTED_STATUSES)

# Decision date earlier than submission date (only checkable when both dates are valid).
decided_early = claims["decided_dt"] < claims["submitted_dt"]

# Add the marks to the table so you can filter on them.
claims["flag_missing"] = any_missing
claims["flag_wrong_type"] = any_wrong_type
claims["flag_unexpected_category"] = bad_category
claims["flag_unexpected_status"] = bad_status
claims["flag_decided_before_submitted"] = decided_early
flag_cols = [c for c in claims.columns if c.startswith("flag_")]
claims["has_problem"] = claims[flag_cols].any(axis=1)

# A short plain-English note per row.
labels = {"flag_missing": "missing value", "flag_wrong_type": "wrong type",
          "flag_unexpected_category": "unexpected category",
          "flag_unexpected_status": "unexpected status",
          "flag_decided_before_submitted": "decided before submitted"}
notes = pd.Series("", index=claims.index)
for flag, label in labels.items():
    notes = notes.mask(claims[flag], notes + label + "; ")
claims["problem_notes"] = notes.str.rstrip("; ")

# --- Reports ---
print(f"Total rows: {len(claims)}  (all kept)\n")

summary = pd.DataFrame({
    "Problem": ["Missing value (any column)", "Wrong type (any column)", "Unexpected Category",
                "Unexpected Status", "Decided before Submitted", "Any problem"],
    "Rows affected": [int(any_missing.sum()), int(any_wrong_type.sum()), int(bad_category.sum()),
                      int(bad_status.sum()), int(decided_early.sum()), int(claims["has_problem"].sum())],
})
display(summary)

print("Missing values, by column:")
display(pd.DataFrame({"Rows missing": {c: int(m.sum()) for c, m in missing_by_col.items()}}))

print("Wrong type, by column (value present but not a valid number or date):")
display(pd.DataFrame({"Rows affected": {"Amount (not a number)": int(bad_amount.sum()),
                                        "Submitted (not a date)": int(bad_submitted.sum()),
                                        "Decided (not a date)": int(bad_decided.sum())}}))

print("Unexpected Category values found:")
display(claims.loc[bad_category, "Category"].value_counts().rename("Rows").to_frame())
print("Unexpected Status values found:")
display(claims.loc[bad_status, "Status"].value_counts().rename("Rows").to_frame())

Total rows: 200  (all kept)



,Problem,Rows affected
0,Missing value (any column),26
1,Wrong type (any column),0
2,Unexpected Category,0
3,Unexpected Status,0
4,Decided before Submitted,0
5,Any problem,26


Missing values, by column:


,Rows missing
Claim ID,0
Submitted,0
Employee,0
Employee ID,0
Department,0
Category,0
Amount,0
Status,0
Decided,26


Wrong type, by column (value present but not a valid number or date):


,Rows affected
Amount (not a number),0
Submitted (not a date),0
Decided (not a date),0


Unexpected Category values found:


,Rows
Category,


Unexpected Status values found:


,Rows
Status,


In [5]:
# Peek at the marked rows (first 20). Filter `claims[claims["has_problem"]]` to see them all.
display(claims.loc[claims["has_problem"], ["Claim ID", "Department", "Category", "Amount", "Status",
                                          "Submitted", "Decided", "problem_notes"]].head(20))

,Claim ID,Department,Category,Amount,Status,Submitted,Decided,problem_notes
12,CLM-00090,Operations,Meals,95.14,Rejected,2026-04-28,NaN,missing value
15,CLM-00040,Marketing,Equipment,218.17,Approved,2026-09-04,NaN,missing value
21,CLM-00137,Operations,Travel,143.48,Approved,2026-02-17,NaN,missing value
24,CLM-00001,Operations,Meals,125.14,Approved,2026-08-06,NaN,missing value
30,CLM-00023,IT,Equipment,73.97,Pending,2026-09-17,NaN,missing value
37,CLM-00168,Sales,Travel,624.88,Pending,2026-01-21,NaN,missing value
49,CLM-00125,Operations,Office Supplies,61.07,Approved,2026-09-01,NaN,missing value
50,CLM-00073,Operations,Equipment,19.16,Approved,2026-01-31,NaN,missing value
66,CLM-00102,Marketing,Travel,116.83,Pending,2026-06-28,NaN,missing value
68,CLM-00089,Sales,Meals,1547.53,Approved,2026-03-18,NaN,missing value


## 3. Total approved spend per department

In [6]:
# Keep approved claims that have a usable Amount and a Department.
q1_rows = claims[(claims["Status"] == "Approved") & claims["amount_num"].notna() & claims["Department"].notna()]

# Add up Amount for each department.
q1 = (q1_rows.groupby("Department")["amount_num"].agg(["sum", "count"])
      .rename(columns={"sum": "Approved spend", "count": "Approved claims"})
      .sort_values("Approved spend", ascending=False)
      .round(2))
display(q1)

fig = px.bar(q1.reset_index(), x="Department", y="Approved spend", text_auto=",.0f",
             title="Total approved spend per department")
fig.show()

print(f"How: sum of Amount where Status = Approved, grouped by Department. Rows used: {len(q1_rows)} of {len(claims)}.")

,Approved spend,Approved claims
Department,,
Operations,20784.32,48
IT,18330.21,34
Sales,13174.49,28
Finance,13036.41,26
Marketing,10947.75,26


How: sum of Amount where Status = Approved, grouped by Department. Rows used: 162 of 200.


## 4. Rejection rate per category

In [7]:
# Only decided claims count: Approved or Rejected (Pending is left out).
q2_rows = claims[claims["Status"].isin(["Approved", "Rejected"]) & claims["Category"].notna()]

# Count approved and rejected per category.
q2 = pd.crosstab(q2_rows["Category"], q2_rows["Status"]).reindex(columns=["Approved", "Rejected"], fill_value=0)
q2["Rejection rate"] = q2["Rejected"] / (q2["Approved"] + q2["Rejected"])
q2 = q2.sort_values("Rejection rate", ascending=False)

# Show the rate as a percentage in the table.
display(q2.assign(**{"Rejection rate": q2["Rejection rate"].map("{:.1%}".format)}))

fig = px.bar(q2.reset_index(), x="Category", y="Rejection rate", text_auto=".1%",
             title="Rejection rate per category")
fig.update_yaxes(tickformat=".0%")
fig.show()

print(f"How: Rejected / (Approved + Rejected) per Category, Pending ignored. Rows used: {len(q2_rows)} of {len(claims)}.")

Status,Approved,Rejected,Rejection rate
Category,,,
Equipment,26,5,16.1%
Travel,33,4,10.8%
Office Supplies,28,3,9.7%
Meals,33,3,8.3%
Software,42,1,2.3%


How: Rejected / (Approved + Rejected) per Category, Pending ignored. Rows used: 178 of 200.


## 5. Claims above their category's policy limit

In [8]:
# Keep claims with a usable Amount and a category that has a policy limit.
q3_rows = claims[claims["amount_num"].notna() & claims["Category"].isin(POLICY_LIMITS.keys())].copy()

# Compare each Amount with its category's limit (equal to the limit is not above it).
q3_rows["limit"] = q3_rows["Category"].map(POLICY_LIMITS)
q3_rows["over_limit"] = q3_rows["amount_num"] > q3_rows["limit"]

q3 = pd.DataFrame({"Limit": pd.Series(POLICY_LIMITS)})
q3["Claims checked"] = q3_rows.groupby("Category").size().reindex(q3.index, fill_value=0)
q3["Claims above limit"] = q3_rows.groupby("Category")["over_limit"].sum().reindex(q3.index, fill_value=0).astype(int)
display(q3)

print(f"Total claims above their category limit: {int(q3_rows['over_limit'].sum())}")

fig = px.bar(q3.reset_index(names="Category"), x="Category", y="Claims above limit", text_auto=True,
             title="Claims above policy limit, by category")
fig.show()

skipped = len(claims) - len(q3_rows)
print(f"How: count of claims where Amount is greater than the category limit. Rows used: {len(q3_rows)} of {len(claims)} ({skipped} skipped: category without a limit or Amount not usable).")

,Limit,Claims checked,Claims above limit
Travel,1500,43,1
Meals,75,40,32
Software,500,49,17
Equipment,1200,36,3
Office Supplies,200,32,19


Total claims above their category limit: 72


How: count of claims where Amount is greater than the category limit. Rows used: 200 of 200 (0 skipped: category without a limit or Amount not usable).
